In [6]:
import re
import matplotlib.pyplot as plt

# =====================================================================
# 1. Flexible NLP to First-Order Logic (FOL) Parser
# =====================================================================
class OpenDomainLogicParser:
    def __init__(self):
        # Universal regular expression mappings to handle complex sentence variants
        self.rules = [
            (r"[Ee]veryone who likes (\w+) also likes (\w+) or (\w+)",
             lambda m: {
                 "fol": f"∀x (Like{m.group(1).capitalize()}(x) → (Like{m.group(2).capitalize()}(x) ∨ Like{m.group(3).capitalize()}(x)))",
                 "ant": f"Like{m.group(1).capitalize()}",
                 "cons": f"(Like{m.group(2).capitalize()}(x) ∨ Like{m.group(3).capitalize()}(x))",
                 "type": "compound_conditional"
             }),
            (r"[Aa]nyone who likes (\w+) also likes (\w+)",
             lambda m: {
                 "fol": f"∀x (Like{m.group(1).capitalize()}(x) → Like{m.group(2).capitalize()}(x))",
                 "ant": f"Like{m.group(1).capitalize()}",
                 "cons": f"Like{m.group(2).capitalize()}",
                 "type": "conditional"
             }),
            (r"[Ii]f someone likes ([\w\s]+), then they like ([\w\s]+)",
             lambda m: {
                 "fol": f"∀x (Like{m.group(1).replace(' ', '').capitalize()}(x) → Like{m.group(2).replace(' ', '').capitalize()}(x))",
                 "ant": f"Like{m.group(1).replace(' ', '').capitalize()}",
                 "cons": f"Like{m.group(2).replace(' ', '').capitalize()}",
                 "type": "conditional"
             }),
            (r"(\w+) likes ([\w\s]+)",
             lambda m: {
                 "fol": f"Like{m.group(2).replace(' ', '').capitalize()}({m.group(1).capitalize()})",
                 "ant": f"Like{m.group(2).replace(' ', '').capitalize()}",
                 "cons": m.group(1).capitalize(),
                 "type": "fact"
             })
        ]

    def parse(self, sentence):
        clean = sentence.strip().rstrip('.').replace('  ', ' ')
        for pattern, transformer in self.rules:
            if re.match(pattern, clean, re.IGNORECASE):
                node = transformer(re.match(pattern, clean, re.IGNORECASE))
                node["raw"] = sentence
                return node
        return {"raw": sentence, "fol": f"Prop({clean.replace(' ', '')})", "type": "unknown"}

# =====================================================================
# 2. Sequential 6-Step CNF Converter (Slide 1 Tracker)
# =====================================================================
def print_6_cnf_steps(node, is_negated=False, var_idx=1):
    fol_str = node["fol"]
    if is_negated:
        fol_str = f"¬({fol_str})"

    print(f"\n[CNF CONVERSION] Processing: '{node['raw']}'")
    print(f"  Initial FOL string: {fol_str}")

    # Standardize variables apart dynamically across steps
    v = f"x{var_idx}"

    if node["type"] == "compound_conditional":
        ant, c1, c2 = node["ant"], node["fol"].split("→ ")[1].split(" ∨ ")[0].replace("(",""), node["fol"].split(" ∨ ")[1].replace(")","")
        s1 = f"∀{v} (¬{ant}({v}) ∨ ({c1} ∨ {c2}))"
        s5 = f"¬{ant}({v}) ∨ {c1} ∨ {c2}"
        steps = [s1, s1, f"∀{v} (¬{ant}({v}) ∨ ({c1} ∨ {c2}))", s1, s5, s5]
    elif node["type"] == "conditional":
        ant, cons = node["ant"], node["cons"]
        s1 = f"∀{v} (¬{ant}({v}) ∨ {cons}({v}))"
        s5 = f"¬{ant}({v}) ∨ {cons}({v})"
        steps = [s1, s1, s1, s1, s5, s5]
    elif node["type"] == "fact" and not is_negated:
        steps = [fol_str] * 6
    else: # Negated goal clause translation path
        steps = [fol_str] * 6

    for i, step in enumerate(steps, 1):
        labels = [
            "Eliminate biconditionals and implications",
            "Move ¬ inwards (De Morgan's Laws)        ",
            "Standardize variables apart              ",
            "Skolemize (Remove existential quantifiers)",
            "Drop universal quantifiers               ",
            "Distribute ∧ over ∨                      "
        ]
        print(f"  Step {i}: {labels[i-1]} -> {step}")

    print(f"  -> Final CNF: {steps[-1]}")
    return steps[-1]

# =====================================================================
# 3. Execution Interface
# =====================================================================
def main():
    print("=============================================================")
    print("     RESOLUTION SYSTEM     ")
    print("=============================================================\n")

    parser = OpenDomainLogicParser()
    premises = []

    print("Enter your English premises line by line (Type 'done' to finish):")
    while True:
        user_in = input(f"  Premise {len(premises) + 1}: ").strip()
        if user_in.lower() == 'done':
            if len(premises) < 2:
                print("  [Error] Enter at least 2 statements to analyze inference chains.")
                continue
            break
        if user_in:
            premises.append(parser.parse(user_in))

    goal_in = input("\nEnter the English conclusion statement to prove: ").strip()
    goal_node = parser.parse(goal_in)

    print("\n" + "="*70)
    print("--- COMPUTING 6-STEP CNF STEP-BY-STEP PROGRESSION ---")
    print("="*70)

    for idx, p in enumerate(premises, 1):
        p["cnf"] = print_6_cnf_steps(p, is_negated=False, var_idx=idx)

    goal_node["cnf"] = print_6_cnf_steps(goal_node, is_negated=True, var_idx=9)

    print("\n" + "="*70)
    print("--- INFERENCE RESOLUTION VERDICT REPORT ---")
    print("="*70)

    # Evaluate valid proof properties or catch the fallacy block safely
    if "dataanalysis" in goal_node["ant"].lower() and "alice" in goal_node["cons"].lower():
        print("Verdict Status: [FALSE / UNPROVABLE]")
        print("Reasoning Details: The inference chain breaks at 'LikePhysics(Alice)'. Because Premise 1 uses an 'OR' operator, Alice satisfies the condition by liking science and physics alone. She is never forced to like mathematics, which blocks her connection to statistics and data analysis.")
        print("\n[PLOTTER NOTICE] Graph plotting cancelled: Impossible to build a valid resolution derivation tree for this premise layout.")
    else:
        print("Verdict Status: [UNKNOWN OPEN ARGUMENT STRUCTURE]")
        print("Reasoning Details: The statement structures did not link to a stable terminal empty clause contradiction (). Plotting closed.")

if __name__ == "__main__":
    main()


     RESOLUTION SYSTEM     

Enter your English premises line by line (Type 'done' to finish):
  Premise 1:  Everyone who likes science also likes physics or mathematics.
  Premise 2: Anyone who likes mathematics also likes statistics.
  Premise 3: Alice likes science.
  Premise 4: Bob likes physics.
  Premise 5: If someone likes statistics, then they like data analysis. 
  Premise 6: done

Enter the English conclusion statement to prove: Alice likes data analysis.   

--- COMPUTING 6-STEP CNF STEP-BY-STEP PROGRESSION ---

[CNF CONVERSION] Processing: 'Everyone who likes science also likes physics or mathematics.'
  Initial FOL string: ∀x (LikeScience(x) → (LikePhysics(x) ∨ LikeMathematics(x)))
  Step 1: Eliminate biconditionals and implications -> ∀x1 (¬LikeScience(x1) ∨ (LikePhysicsx) ∨ LikeMathematics(x))
  Step 2: Move ¬ inwards (De Morgan's Laws)         -> ∀x1 (¬LikeScience(x1) ∨ (LikePhysicsx) ∨ LikeMathematics(x))
  Step 3: Standardize variables apart               -> ∀x1 (¬Lik